In [3]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F



In [ ]:
tokenizor=BertTokenizer.from_pretrained("bert-base-uncased")
model=BertForMaskedLM.from_pretrained("bert-base-uncased")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000218264E0050>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 9971e5e1-29ff-413a-b9c6-3569043cd26c)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000021808F4F790>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 80bd7494-f495-48d2-95ef-ee531ee6789d)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/toke

model.safetensors:   5%|4         | 21.0M/440M [00:00<?, ?B/s]

In [ ]:
text="The capital city of france is [Mask]"

In [ ]:
input=tokenizor(text,return_tensors="pt")

In [ ]:
with torch.inference_mode():
    logits=model(**input).logits

In [ ]:
mask_idx=(input.input_ids == tokenizor.mask_token_id).nonzero(as_tuple=True)[1]

In [ ]:
probs=F.softmax(logits[0,mask_idx],dim=-1)
top_k=torch.topk(probs,k=5,dim=-1)

In [ ]:
print("Top 5 predictions for [Mask]")
for token_id,prob in zip(top_k.indices[0],top_k.values[0]):
    token=tokenizor.decode([token_id])
    print(f"{token:<12}->{prob.item():.4f}")